# Step 2.13: trade paths

This notebook is the check of step 2.13. Every backtest now stores, for each trade, the compact path of the price its position is marked at, which the prop simulator of Phase 7 replays without running the strategy again (`docs/masterplan/03-prop-simulator.md` §4.2): each fill at its own price, each new favourable extreme, each new adverse extreme since the last favourable one, the mark at every whole minute while the trade is open, and, on bars, the adverse extreme a bar may have reached after a new favourable one. `run.trade_paths` holds them, a row per point, `trade` its trade's row in `run.trades`, every price in the trade's entry contract's prices; `run.json` counts the trades by the fidelity of their paths, which `run.report()` prints.

The notebook generates five years of ES sessions by seed, opens the paths of a run on your MES 1-minute bars, picks its losing trade that went furthest against it, and plots that trade's path: its most adverse point is the trade's MAE, and it ends at the trade's exit price. Then it measures what paths cost on the five years, for a strategy that holds minutes and one that holds days: the size of `trade_paths.parquet` beside the run's other files, and the engine's time with and without paths. The years go into your `FEFAT_HOME` cache folder and may be deleted at any time. Pick the `.venv` kernel and run all cells.

## Five generated years

ESH2026's sessions from 2021-03-22 to 2026-03-19, the trade dates of CME's equity index calendar, each as dense as the committed fixtures. `scripts/gen_fixtures.py` writes the 1-minute bars of every session, and the trades and 1-second bars of the last session alone, into `<FEFAT_HOME>/cache/synthetic-es-2021-03-22-2026-03-19/`. Generating them takes about seven minutes the first time; later the cell reuses them. The runs below fill on the 1-minute bars, as a store of 1-minute bars alone, such as your MES, fills them.

Expected on this PC, the first time: `1,258 sessions from 2021-03-22 to 2026-03-19, ready in 430 s`, give or take; later runs print the same line with a time of 0 s.

In [ ]:
import os
import subprocess
import sys
import time
from contextlib import contextmanager
from datetime import date, timedelta
from pathlib import Path

import fefat_engine
import matplotlib.pyplot as plt
import polars as pl

import fefat as ff
from fefat.backtest import RunConfig
from fefat.backtest.run_inputs import run_inputs
from fefat.config import resolve_paths
from fefat.market.calendar import counted
from fefat.market.fidelity import ONE_MINUTE
from fefat.market.saved_instruments import instrument_table

REPOSITORY = Path(ff.__file__).resolve().parents[4]
paths = resolve_paths(os.environ)
calendar = instrument_table(paths.data_dir).sessions_of("ES", paths.cache_dir)
sessions = calendar.between(date(2021, 3, 22), date(2026, 3, 19))
sessions = sessions.get_column("trade_date").to_list()
sessions = sessions[-int(os.environ.get("FEFAT_CHECK_DAYS", len(sessions))) :]
FIRST, LAST = sessions[0], sessions[-1]
years = paths.cache_dir / f"synthetic-es-{FIRST}-{LAST}"
years_store = years / "data"
started = time.perf_counter()
if not (years_store / "synthetic" / "MANIFEST.json").is_file():
    command = [
        sys.executable,
        str(REPOSITORY / "scripts" / "gen_fixtures.py"),
        "--out",
        str(years_store / "synthetic"),
        "--dates",
        f"{FIRST}..{LAST}",
        "--contracts",
        "ESH2026",
        "--ticks-from",
        str(LAST),
    ]
    generated = subprocess.run(command, capture_output=True, text=True, check=False)
    if generated.returncode != 0:
        raise RuntimeError(f"Expected gen_fixtures.py to pass, got: {generated.stderr}")
print(
    f"{counted(len(sessions), 'session')} from {FIRST} to {LAST}, "
    f"ready in {time.perf_counter() - started:.0f} s"
)

## The trade paths of a run on your MES

The breakout of the earlier checks, a long on each 5-minute close above the highest of the 12 closes before it, with a stop of 12 ticks and a target of 24, over the last 30 days of your MES 1-minute bars, from the provider that holds them. Every day fills on 1-minute bars, so every trade's path has the fidelity `1m`. Without MES in your store, one line says so and the run uses the last 30 days of the generated years instead.

Expected: the run's trades counted by the fidelity of their paths, all `1m`, then the losing trade the cell picks: its row in `run.trades`, its side, the trade date of its entry, what closed it and when, its MAE in ticks, and how many points its path has, of each kind.

In [ ]:
STOP_TICKS = 12


def breakout_on(root):
    """A long on a 5-minute close above the highest of the 12 closes before it, with a stop of
    `STOP_TICKS` and a target twice as far."""
    return ff.SignalStrategy(
        instrument=root,
        timeframe="5m",
        features={"highest": pl.col("close").rolling_max(12).shift(1)},
        long_entry=pl.col("close") > pl.col("highest"),
        short_entry=pl.lit(False),
        execution=ff.Execution(
            entry=ff.Market(),
            stop=ff.Stop(ticks=STOP_TICKS),
            targets=[ff.Target(ticks=2 * STOP_TICKS)],
            sizing=ff.Fixed(1),
        ),
    )


@contextmanager
def store_at(folder):
    """`FEFAT_DATA_DIR` pointed at `folder` while the block runs, then back at yours."""
    yours = os.environ.get("FEFAT_DATA_DIR")
    os.environ["FEFAT_DATA_DIR"] = str(folder)
    try:
        yield
    finally:
        if yours is None:
            os.environ.pop("FEFAT_DATA_DIR")
        else:
            os.environ["FEFAT_DATA_DIR"] = yours


listed = ff.market.datasets().filter(pl.col("root") == "MES", pl.col("schema") == "ohlcv-1m")
if listed.is_empty():
    print(
        "No MES 1-minute bars in your store: the run reads the generated years' last 30 days "
        "instead (import MES, docs/data-guide.md, to open a run on your own bars)."
    )
    with store_at(years_store):
        run = ff.backtest.run(
            breakout_on("ES"),
            max(FIRST, LAST - timedelta(days=30)),
            LAST,
            RunConfig(fidelity=ONE_MINUTE),
        )
else:
    mes = listed.row(0, named=True)
    last = mes["last_trade_date"]
    run = ff.backtest.run(
        breakout_on("MES"), last - timedelta(days=30), last, RunConfig(provider=mes["provider"])
    )
for path_fidelity in run.record.path_fidelities:
    print(f"Paths of {path_fidelity.fidelity}: {counted(path_fidelity.trades, 'trade')}")
trades = run.trades.with_row_index("trade")
losing = trades.filter(pl.col("pnl") < 0).sort("mae_ticks", descending=True, maintain_order=True)
if losing.is_empty():
    raise ValueError(f"Expected a losing trade in run {run.hash[:12]}, got none")
TRADE = losing.row(0, named=True)
path = run.trade_paths.filter(pl.col("trade") == TRADE["trade"])
kinds = path.group_by("kind", maintain_order=True).len()
minutes = (TRADE["exit_ts"] - TRADE["entry_ts"]) / 6e10
print(
    f"Trade {TRADE['trade']}, a {TRADE['side']} entered on {TRADE['entry_trade_date']}, closed by "
    f"its {TRADE['exit_reason']} {minutes:.0f} minutes later, MAE {TRADE['mae_ticks']} ticks: "
    f"{counted(path.height, 'point')}, "
    + ", ".join(f"{count} {kind}" for kind, count in kinds.iter_rows())
)

## The losing trade's path

The plot draws the path's points in their order, in ticks from the trade's entry price, so that no price of your data leaves your machine in a saved output, a screenshot, or a later conversation: each fill, extreme, and minute, the fills marked, with the entry at 0, the level of the trade's MAE, below a long's entry or above a short's, and its exit. The path's most adverse point, the legs of a roll that transfers the trade left out, which no excursion counts, lies its MAE from the entry; and its last point is the fill that closed the trade, at its exit price.

Expected: the plot, then `Its most adverse point lies its MAE of <n> ticks from its entry: True` and `Its path ends at its exit price: True`. The notebook prints and plots no price, only ticks, counts, and dates.

In [ ]:
path = run.trade_paths.filter(pl.col("trade") == TRADE["trade"])
tick = int(run.record.instrument.tick_size.scaleb(9))
sign = 1 if TRADE["side"] == "long" else -1
roles = run.fills.select("order", "role")
points = path.join(roles, on="order", how="left", maintain_order="left").with_row_index("at")
transfer_leg = pl.col("role").str.starts_with("roll_").fill_null(False) & (
    pl.col("at") < points.height - 1
)
market = points.filter(~transfer_leg, pl.col("kind") != "possible_adverse")
prices = market.get_column("price")
worst = prices.min() if sign == 1 else prices.max()
mae_level = TRADE["entry_price"] - sign * TRADE["mae_ticks"] * tick


def ticks_from_entry(price):
    """A price, or a column of them, in ticks from the trade's entry price."""
    return (price - TRADE["entry_price"]) / tick


fills = points.filter(pl.col("kind") == "fill")
figure, axes = plt.subplots(figsize=(10, 4))
path_at = pl.from_epoch(points.get_column("ts"), time_unit="ns")
axes.plot(path_at, ticks_from_entry(points.get_column("price")), linewidth=1, label="path")
fills_at = pl.from_epoch(fills.get_column("ts"), time_unit="ns")
axes.scatter(fills_at, ticks_from_entry(fills.get_column("price")), color="black", label="fills")
axes.axhline(0, color="grey", linestyle=":", label="entry")
mae_label = f"MAE, {TRADE['mae_ticks']} ticks"
axes.axhline(ticks_from_entry(mae_level), color="red", linestyle="--", label=mae_label)
axes.axhline(ticks_from_entry(TRADE["exit_price"]), color="green", linestyle=":", label="exit")
axes.set_title(
    f"Trade {TRADE['trade']} of run {run.hash[:12]}, path fidelity {TRADE['path_fidelity']}"
)
axes.set_ylabel("ticks from the entry")
axes.legend(loc="best")
print(
    f"Its most adverse point lies its MAE of {TRADE['mae_ticks']} ticks from its entry: "
    f"{worst == mae_level}"
)
last = points.row(-1, named=True)
print(
    "Its path ends at its exit price: "
    f"{last['kind'] == 'fill' and last['price'] == TRADE['exit_price']}"
)

## What paths cost

Done means 3 of the step: on the five generated years, the breakout above, which holds minutes, and a strategy that holds days, a long on a 5-minute close above the highest of the 288 closes before it, a day of bars, with a stop of 400 ticks and no target, closed on a close below the lowest of the 288 before it. Each runs once through `ff.backtest.run` at 1-minute fidelity and is saved; the cell prints its trades, how long they are held, its path's points, and the size of each file of its folder that matters here, `trade_paths.parquet` beside `fills`, `trades`, `equity`, and `signals`. Then it runs the engine alone three times with paths and three times without, and prints the fastest of each.

Expected on this PC, the times varying with what else runs: `breakout: 17,333 trades, held 0.9 h on average, 74.4 h at most; 2,560,581 path points`, with 11.47 MB of `trade_paths` beside 0.49 MB of `fills` and 0.65 MB of `trades`, and the engine about 0.13 s slower with paths than its 0.62 s without; then `days-long: 394 trades, held 56.1 h on average, 296.0 h at most; 3,522,426 path points`, with 15.93 MB of paths, and the engine about 0.13 s slower than its 0.46 s without. On 1-minute bars each minute restarts the tracking, so that a bar's extremes are new points of its minute, about 2.7 points a minute in a position; a multi-day hold also carries a mark at every minute it is open, nights and weekends included, which is what makes its paths larger (`03` §4.2).

In [ ]:
def days_long_on(root):
    """A long on a 5-minute close above the highest of the 288 closes before it, a stop of 400
    ticks and no target, closed on a close below the lowest of the 288 before it."""
    return ff.SignalStrategy(
        instrument=root,
        timeframe="5m",
        features={
            "highest": pl.col("close").rolling_max(288).shift(1),
            "lowest": pl.col("close").rolling_min(288).shift(1),
        },
        long_entry=pl.col("close") > pl.col("highest"),
        short_entry=pl.lit(False),
        long_exit=pl.col("close") < pl.col("lowest"),
        execution=ff.Execution(entry=ff.Market(), stop=ff.Stop(ticks=400), sizing=ff.Fixed(1)),
    )


def megabytes(path):
    return f"{path.stat().st_size / 1e6:.2f} MB"


on_minutes = RunConfig(fidelity=ONE_MINUTE)
with store_at(years_store):
    for name, strategy in (("breakout", breakout_on("ES")), ("days-long", days_long_on("ES"))):
        saved = ff.backtest.run(strategy, FIRST, LAST, on_minutes)
        held = (saved.trades.get_column("exit_ts") - saved.trades.get_column("entry_ts")) / 3.6e12
        files = ", ".join(
            f"{artifact} {megabytes(saved.folder / f'{artifact}.parquet')}"
            for artifact in ("trade_paths", "fills", "trades", "equity", "signals")
        )
        inputs = run_inputs(strategy, FIRST, LAST, on_minutes, resolve_paths(os.environ))
        caches = [str(cache) for cache in inputs.cache_paths(resolve_paths(os.environ))]
        engine = (inputs.spec_json(), inputs.signals, caches, inputs.config_json(), "skipping")
        seconds = {}
        for kept in (True, False):
            times = []
            for _ in range(3):
                started = time.perf_counter()
                fefat_engine.run_signal_backtest(*engine, trade_paths=kept)
                times.append(time.perf_counter() - started)
            seconds[kept] = min(times)
        how_long = "none held"
        if not saved.trades.is_empty():
            how_long = f"held {held.mean():.1f} h on average, {held.max():.1f} h at most"
        print(
            f"{name}: {counted(saved.trades.height, 'trade')}, {how_long}; "
            f"{counted(saved.trade_paths.height, 'path point')}"
        )
        print(f"  files: {files}")
        print(
            f"  engine with paths {seconds[True]:.2f} s, without {seconds[False]:.2f} s, the "
            "fastest of three"
        )

## Your check

Open the trade paths of one real run, pick a losing trade, and plot its path: its most adverse point matches the trade's MAE in the trades table, and the path ends at the trade's exit price. The two `True` lines under the plot say so for the trade the notebook picked; pick another with `TRADE = trades.row(<row>, named=True)` and run the plot's cell again.

Read the plot as the prop simulator will: each fill restarts the tracking at its own price, each whole minute holds the price the position was marked at then, and the extremes between them, in order, are what a prop rule's threshold can be crossed at. On 1-minute bars the order inside a bar is the engine's assumption, the adverse extreme first; a `possible_adverse` point after a new favourable extreme marks a bar whose other order would have fallen further from that peak.